# 10. Diagrams meet CAD: M1 and M0

Tutorial 6's structure diagram and tutorial 7's 3D viewer, wired so a
click in either one lights up the other. The wiring is fun; the point
is what the round trip *teaches*: the two panes live on different
levels of the modeling stack. The diagram draws **M1**, the model --
one `rotors : Rotor [4]` node, a description with a multiplicity. The
3D scene renders **M0**, an interpretation -- four actual motor
meshes, one per individual. Cross-selection makes the level jump
visible in both directions.

You will learn how to:

- read `rotors : Rotor [4]` as *one* M1 element, not four;
- build the M0 population with `m0.interpret` and roll up mass over
  the individuals that exist (tutorial 9's machinery);
- bake per-instance geometry (`drone_geometry(split_instances=True)`)
  and key each mesh part to an M0 individual id;
- watch the M1 -> M0 **fan-out**: selecting the one usage highlights
  all four instance meshes;
- watch the M0 -> M1 **projection**: picking one motor selects the one
  diagram node, while `on_pick` reports *which* individual was hit.

Prerequisites: tutorial 6 (diagrams, `on_select`) and tutorial 9
(`longeron.m0`). The widgets need the `viz` extra and JupyterLab for
pixels, but every cell below also runs headless -- browser clicks are
simulated by writing the same traitlets the front-ends write, and
assertions after each step prove the claims.

In [ ]:
import json

import ipywidgets as W

import longeron
from longeron import diagrams, m0
from longeron.analysis import geometry, link, viewer3d

model = longeron.load("../examples/drone.sysml")

## M1: one usage, whatever the multiplicity says

`Drone::QuadCopter` owns `part rotors : Rotor[4]`. At M1 that is a
*single* element -- a description saying "four of these exist", not
four of anything. The structure diagram is an M1 view, so it draws
exactly one `rotors` node; there is nothing in the model (and so
nothing in the diagram) for an individual rotor to be.

In [ ]:
rotors = model.find("Drone::QuadCopter::rotors")
print(f"{rotors.qualified_name} : {rotors.types[0]} [{rotors.multiplicity.upper}]")

# one element, multiplicity four: the "4" is data on the description
assert sum(1 for e in model.iter_tree() if e.name == "rotors") == 1
assert rotors.multiplicity.upper.value == 4

## M0: the interpretation has four, each with a name

`m0.interpret` builds the population the description denotes: four
`Rotor` **individuals** with stable `qname#index` ids, plus the
chassis and battery singletons. Roll-ups run over the individuals that
actually exist -- `sum(rotors.mass)` adds four real masses, where the
M1 `totalMass` build-up hand-encodes `4.0 * 0.06`. Tutorial 9 is the
deep dive; here the population is the cast of characters for the 3D
scene.

In [ ]:
quad = m0.interpret(model, "Drone::QuadCopter")
for individual in quad.individuals():
    print(individual.id)

rotor_ids = [rotor.id for rotor in quad.root.slots["rotors"]]
assert rotor_ids == [f"Drone::QuadCopter#0.rotors#{i}" for i in range(4)]
assert quad.rollup("sum(rotors.mass)") == 4 * 0.06  # over actual individuals
print("\nsum(rotors.mass) over the population:", quad.rollup("sum(rotors.mass)"), "kg")

## The 3D scene is a rendering of the M0 population

`drone_geometry` normally merges each part kind into one mesh (one
draw call each). `split_instances=True` keeps the motor and prop
instances separate -- `motor1` .. `motor4`, `prop1` .. `prop4`, the
same children the cadquery assembly exports -- so each can carry its
own identity key. The part map stamps **M0 individual ids** as those
keys: a `motor`/`prop` pair renders one rotor individual (a motor can
and its prop disk are two views of the same rotor), the frame renders
the chassis individual, and the ESC has no model part and stays
untagged (inert). Sizes still come from the model's own catalog
values.

The rule that ties an individual key back to M1 is
`link.individual_qname`: strip each dotted segment's `#index` and join
with `::` -- so `Drone::QuadCopter#0.rotors#2` *belongs to* the usage
`Drone::QuadCopter::rotors`. That one derivation is the whole
M0 -> M1 projection.

In [ ]:
interp = longeron.Interpreter(model)
mesh = geometry.drone_geometry(
    prop_diameter_in=9.0,  # geometry-only: not an attribute of this model
    motor_mass=interp.evaluate("Drone::Rotor::mass"),
    battery_mass=interp.evaluate("Drone::Battery::mass"),
    esc_mass=0.012,  # the 30.5 mm stack heuristic; no ESC in the model
    split_instances=True,
)
print([part["name"] for part in mesh["parts"]])

PART_MAP = {
    "frame": quad.root.slots["chassis"].id,
    "battery": quad.root.slots["battery"].id,
    **{f"motor{i + 1}": rotor_id for i, rotor_id in enumerate(rotor_ids)},
    **{f"prop{i + 1}": rotor_id for i, rotor_id in enumerate(rotor_ids)},
}

assert link.individual_qname("Drone::QuadCopter#0.rotors#2") == "Drone::QuadCopter::rotors"
assert link.individual_qname("Drone::QuadCopter::rotors") is None  # not an individual id

## Side by side, cross-linked

`link_selection` stamps the keys onto the viewer's mesh and wires both
directions; it returns `unlink` for disposal. `on_pick` is the M0 tap:
it receives every raycaster report exactly as written -- the picked
individual id, or `[]` for a background click -- *before* the
selection is projected to M1, so nothing the diagram cannot represent
gets lost.

**Try it in JupyterLab:** click `rotors` in the diagram and watch all
four motors pop; then click a single motor can and watch the diagram
select the one `rotors` node while `picked` records the individual.

In [ ]:
HEIGHT = 650

structure = diagrams.structure_diagram(model, height=f"{HEIGHT}px")  # match the 3D viewer's span
viewer = viewer3d.mesh_viewer(
    mesh, label="Drone::QuadCopter -- one M0 interpretation", width_px=HEIGHT, height_px=HEIGHT
)

picked: list = []
unlink = link.link_selection(structure, viewer, model, part_map=PART_MAP, on_pick=picked.append)

# the viewer's stage is responsive: its rendered height is containerWidth /
# aspect, so a FIXED container width pins it at exactly height_px (480);
# the diagram takes the remaining width at the same fixed height
viewer.layout = W.Layout(width=f"{HEIGHT}px", flex="0 0 auto")
structure.layout.width = "auto"
structure.layout.flex = "1 1 auto"
combined = W.HBox([structure, viewer], layout=W.Layout(align_items="stretch", width="100%", overflow="hidden"))
combined

## M1 -> M0: the fan-out, made visible

Selecting the *one* `rotors` node highlights *four* motor meshes
(eight parts counting the prop disks): a usage matches every key that
derives from it. Selecting the `Rotor` definition reaches the same
individuals through the usage it types, and selecting the whole
`QuadCopter` matches the entire rendered population. The cells below
drive the same traitlets a browser click writes, so the claims hold
headless.

In [ ]:
# a click on the one M1 usage -> all four M0 individuals light up
structure.view.selection.ids = ["Drone::QuadCopter::rotors"]
assert json.loads(viewer.highlight_json) == sorted(rotor_ids)

# the definition reaches the same population through the usage it types
structure.view.selection.ids = ["Drone::Rotor"]
assert json.loads(viewer.highlight_json) == sorted(rotor_ids)

# the whole assembly -> chassis + battery + all four rotors
structure.view.selection.ids = ["Drone::QuadCopter"]
assert json.loads(viewer.highlight_json) == sorted(set(PART_MAP.values()))

# something not rendered clears rather than dims
structure.view.selection.ids = ["Drone::HoverTime"]
assert viewer.highlight_json == "[]"

## M0 -> M1: a projection that keeps the individual

The reverse direction is many-to-one: the diagram has no node for
`rotors#2`, so picking the third motor can selects the one `rotors`
usage -- and the forward direction immediately fans back out, so all
four instances stay lit. The pick itself is not lost: `on_pick` got
the individual id, ready for an M0-side reaction -- look the
individual up in the population, trace it, log it.

In [ ]:
# what the canvas raycaster writes when you click the third motor can
viewer.picked_json = json.dumps(["Drone::QuadCopter#0.rotors#2"])
assert list(structure.view.selection.ids) == ["Drone::QuadCopter::rotors"]  # M1
assert json.loads(viewer.highlight_json) == sorted(rotor_ids)  # ... fans back out
assert picked[-1] == ["Drone::QuadCopter#0.rotors#2"]  # M0: the individual, kept

individual = next(i for i in quad.individuals() if i.id == picked[-1][0])
print("picked:", individual, " mass:", individual.slots["mass"], "kg")

# a background click clears both panes and reports [] on the tap
viewer.picked_json = json.dumps([])
assert picked[-1] == []
assert list(structure.view.selection.ids) == []
assert viewer.highlight_json == "[]"

# leave the fan-out visible for a live front-end
structure.view.selection.ids = ["Drone::QuadCopter::rotors"]
assert json.loads(viewer.highlight_json) == sorted(rotor_ids)
print("M1 <-> M0 round trip: all assertions passed")

## Which plane to think in

Think at **M1** when the question is about the *design*: architecture
and trades quantify over usages and definitions -- swap the rotor
definition, re-run the study, compare mixes (tutorials 6-7). Think at
**M0** when the question is about a *population*: roll-ups that weigh
what actually exists, per-individual identities and traces,
Monte-Carlo over drawn configurations (tutorial 9). The linked panes
above are the two planes side by side: the diagram can only ever
select descriptions, the scene renders one interpretation of them, and
`individual_qname` is the entire bridge.

Headless caveat: everything above ran without a browser because both
front-ends are pure painters over synced traitlets. The pixels -- the
emissive pop, the dimming, a real raycast from a canvas click -- need
JupyterLab (`pixi run lab`).